# GemmaFit Phase 3 — Algorithm Validation Notebook

**Purpose**: Validate the multi-exercise pipeline on a real video, end to end.

Pipeline:
```
Video → MediaPipe Pose → Landmarks (33×3)
      → Joint Angles → Exercise Detection → Applicability Gates
      → Template Metrics → Structured Motion Report
      → mock_gemma_feedback / Gemma 4 (optional)
```

Visualizations produced:
1. Skeleton overlay on sampled frames
2. Joint angle time series (knee / hip / spine / elbow)
3. 2D motion trajectories (knee, wrist, COM) with fading trails
4. Exercise detection confidence over time
5. Quality gate timeline (CRITICAL / WARNING / MONITOR / OK)
6. Single-frame structured report + coaching message

## 1. Setup

In [ ]:
import os, sys, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.patches import Polygon
from matplotlib.collections import LineCollection

import cv2
import mediapipe as mp

PROTO_DIR = Path(os.getcwd()).resolve()
if 'prototype' not in str(PROTO_DIR).lower():
    PROTO_DIR = PROTO_DIR / 'prototype'
ROOT_DIR = PROTO_DIR.parent
sys.path.insert(0, str(PROTO_DIR))

from compute_angles import get_squat_phase
from movement_classifier import KEYPOINT
from com_tracker_prototype import track_com
from rep_counter import RepCounter

from exercises.core import (
    TEMPLATES, detect_exercise, apply_gates, extract_template_metrics,
    build_report, mock_gemma_feedback,
    STATUS_OK, STATUS_MONITOR, STATUS_WARNING, STATUS_CRITICAL,
    STATUS_NOT_APPLICABLE, STATUS_LOW_CONFIDENCE, STATUS_VIEW_LIMITED,
)

plt.rcParams.update({
    'figure.facecolor': '#0f1724', 'axes.facecolor': '#0f1724',
    'savefig.facecolor': '#0f1724',
    'text.color': '#e0e0e0', 'axes.labelcolor': '#e0e0e0',
    'xtick.color': '#a0a0a0', 'ytick.color': '#a0a0a0',
    'axes.edgecolor': '#444', 'axes.titlecolor': '#00D4AA',
    'axes.grid': True, 'grid.color': '#222', 'grid.linewidth': 0.5,
})
print('OK | proto:', PROTO_DIR, '| templates:', list(TEMPLATES.keys()))

## 2. Choose video

In [ ]:
ASSET_DIR = ROOT_DIR / 'test_assets' / 'videos'
AVAILABLE = {
    'squat':    ASSET_DIR / 'squat_wikimedia_01.webm',
    'pushup':   ASSET_DIR / 'pushup_cdc_01.webm',
    'deadlift': ASSET_DIR / 'deadlift_demo.webm',
    'lunge':    ASSET_DIR / 'lunge_kettlebell.webm',
}

VIDEO_KEY = 'squat'         # ← change to 'pushup' / 'deadlift' / 'lunge'
SAMPLE_EVERY = 2             # ← higher = faster but coarser trajectories

VIDEO_PATH = AVAILABLE[VIDEO_KEY]
assert VIDEO_PATH.exists(), f'Missing video: {VIDEO_PATH}'
print('Selected:', VIDEO_PATH.name)

## 3. Extract MediaPipe landmarks

In [ ]:
LANDMARK_NAMES = [
    'nose','left_eye_inner','left_eye','left_eye_outer',
    'right_eye_inner','right_eye','right_eye_outer',
    'left_ear','right_ear','mouth_left','mouth_right',
    'left_shoulder','right_shoulder','left_elbow','right_elbow',
    'left_wrist','right_wrist','left_pinky','right_pinky',
    'left_index','right_index','left_thumb','right_thumb',
    'left_hip','right_hip','left_knee','right_knee',
    'left_ankle','right_ankle','left_heel','right_heel',
    'left_foot_index','right_foot_index',
]

def extract_landmarks(video_path, sample_every=1):
    """Returns: list[dict landmarks], fps, list[BGR frames sampled]."""
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frames_data, raw_frames = [], []
    mp_pose = mp.solutions.pose
    with mp_pose.Pose(min_detection_confidence=0.5,
                       min_tracking_confidence=0.5) as pose:
        idx = 0
        while cap.isOpened():
            ok, img = cap.read()
            if not ok: break
            if idx % sample_every == 0:
                rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                res = pose.process(rgb)
                lm = {}
                if res.pose_landmarks:
                    for i, name in enumerate(LANDMARK_NAMES):
                        if i < len(res.pose_landmarks.landmark):
                            lp = res.pose_landmarks.landmark[i]
                            lm[f'{name}_x']   = lp.x
                            lm[f'{name}_y']   = lp.y
                            lm[f'{name}_z']   = lp.z
                            lm[f'{name}_vis'] = lp.visibility
                frames_data.append({'landmarks': lm, 'frame': idx})
                raw_frames.append(img)
            idx += 1
    cap.release()
    return frames_data, fps, raw_frames

frames, fps, raw = extract_landmarks(VIDEO_PATH, SAMPLE_EVERY)
print(f'Frames sampled: {len(frames)}  |  FPS: {fps:.1f}')

In [ ]:
def lm_to_arr(fd):
    a = np.zeros((33, 3))
    for i, n in enumerate(LANDMARK_NAMES):
        a[i,0] = fd['landmarks'].get(f'{n}_x', 0.0)
        a[i,1] = fd['landmarks'].get(f'{n}_y', 0.0)
        a[i,2] = fd['landmarks'].get(f'{n}_vis', 0.9)
    return a

def lm_to_mp_list(arr):
    class _LM:
        def __init__(self, x, y, z=0.0, vis=0.9):
            self.x=x; self.y=y; self.z=z; self.visibility=vis
    return [_LM(arr[i,0], arr[i,1], 0.0, arr[i,2]) for i in range(33)]

lm_arrs = [lm_to_arr(f) for f in frames]
print('lm_arrs shape:', np.array(lm_arrs).shape)

## 4. Per-frame analysis pipeline

In [ ]:
from movement_classifier import calc_angle

ANGLE_TRIPLETS = {
    'left_knee':   ('left_hip',      'left_knee',     'left_ankle'),
    'right_knee':  ('right_hip',     'right_knee',    'right_ankle'),
    'left_hip':    ('left_shoulder', 'left_hip',      'left_knee'),
    'right_hip':   ('right_shoulder','right_hip',     'right_knee'),
    'left_elbow':  ('left_shoulder', 'left_elbow',    'left_wrist'),
    'right_elbow': ('right_shoulder','right_elbow',   'right_wrist'),
    'spine':       ('left_shoulder', 'left_hip',      'left_knee'),
}

def compute_angles(arr):
    out = {}
    for name, (a, b, c) in ANGLE_TRIPLETS.items():
        try:
            out[name] = float(calc_angle(
                arr[KEYPOINT[a]], arr[KEYPOINT[b]], arr[KEYPOINT[c]]))
        except Exception:
            out[name] = 180.0
    return out

results = []
rep_ctr = RepCounter()
prev_ang = None
for i, arr in enumerate(lm_arrs):
    angles = compute_angles(arr)
    ex     = detect_exercise(arr)
    tmpl   = TEMPLATES.get(ex.exercise, TEMPLATES['unknown'])

    # COM (de Leva)
    com = track_com(lm_to_mp_list(arr), sex='male', contact='bipedal')

    # 8-rule violations (raw, before applicability gate)
    raw_violations = []  # populated by apply_gates internally
    qf, na = apply_gates(raw_violations, tmpl, arr, angles, ex.exercise_confidence)
    metrics = extract_template_metrics(angles, tmpl, arr, fps, prev_ang)

    rep_ctr.update(angles.get('left_knee', 180.0), 0)
    phase = get_squat_phase(angles.get('left_knee', 180.0), 'top')

    report = build_report(i, ex, phase, rep_ctr.rep_count, metrics, qf, na)
    gemma  = mock_gemma_feedback(report)

    results.append({
        'idx': i, 'angles': angles, 'ex': ex, 'tmpl': tmpl,
        'qf': qf, 'na': na, 'metrics': metrics, 'com': com,
        'phase': phase, 'rep': rep_ctr.rep_count,
        'report': report, 'gemma': gemma,
    })
    prev_ang = angles

from collections import Counter
ex_dist = Counter(r['ex'].exercise for r in results)
print('Exercise distribution:', dict(ex_dist))
print('Dominant exercise:    ', ex_dist.most_common(1)[0][0])
print('Mean confidence:      ', f"{np.mean([r['ex'].exercise_confidence for r in results]):.3f}")

## 5. Skeleton overlay on sample frames

In [ ]:
SKELETON = [
    ('left_shoulder','right_shoulder'),
    ('left_shoulder','left_elbow'),('left_elbow','left_wrist'),
    ('right_shoulder','right_elbow'),('right_elbow','right_wrist'),
    ('left_shoulder','left_hip'),('right_shoulder','right_hip'),
    ('left_hip','right_hip'),
    ('left_hip','left_knee'),('left_knee','left_ankle'),
    ('right_hip','right_knee'),('right_knee','right_ankle'),
]

n = min(6, len(results))
sample_idx = np.linspace(0, len(results)-1, n, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, idx in zip(axes.flat, sample_idx):
    arr = lm_arrs[idx]
    img = cv2.cvtColor(raw[idx], cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)
    flagged = {f.joint for f in results[idx]['qf']
               if f.status in (STATUS_CRITICAL, STATUS_WARNING)}
    for a, b in SKELETON:
        ax_, ay_ = arr[KEYPOINT[a],0]*w, arr[KEYPOINT[a],1]*h
        bx_, by_ = arr[KEYPOINT[b],0]*w, arr[KEYPOINT[b],1]*h
        hit = any(j in a or j in b for j in flagged)
        ax.plot([ax_, bx_], [ay_, by_],
                color='#FF4444' if hit else '#4ECDC4', lw=2.5, alpha=0.85)
    for jname in ['left_knee','right_knee','left_hip','right_hip',
                  'left_elbow','right_elbow']:
        kx, ky = arr[KEYPOINT[jname],0]*w, arr[KEYPOINT[jname],1]*h
        ax.scatter([kx],[ky], s=40, c='#00D4AA', edgecolors='white', zorder=5)
    com = results[idx]['com']
    if com:
        ax.scatter([com.com.x*w],[com.com.y*h], s=200,
                   c='#FFD700', marker='*', edgecolors='white', zorder=6)
    ax.set_title(f"f{idx} | {results[idx]['ex'].exercise} "
                 f"({results[idx]['ex'].exercise_confidence:.2f})")
    ax.axis('off')
plt.suptitle(f'Skeleton overlay — {VIDEO_PATH.name}', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Joint angle time series

In [ ]:
joints_to_plot = ['left_knee','right_knee','left_hip','right_hip','spine']
if VIDEO_KEY == 'pushup':
    joints_to_plot = ['left_elbow','right_elbow','spine']

colors = {'left_knee':'#FF6B6B','right_knee':'#4ECDC4',
          'left_hip':'#45B7D1','right_hip':'#96CEB4',
          'spine':'#FFEAA7','left_elbow':'#FFA07A','right_elbow':'#87CEEB'}

fig, ax = plt.subplots(figsize=(13, 4.5))
x = np.arange(len(results))
for j in joints_to_plot:
    y = [r['angles'].get(j, 0.0) for r in results]
    ax.plot(x, y, label=j.replace('_',' ').title(),
            color=colors.get(j,'#888'), lw=1.7)
ax.set_xlabel('Sampled frame index')
ax.set_ylabel('Angle (°)')
ax.set_title(f'Joint angle trends — {VIDEO_PATH.name}')
ax.legend(loc='best', framealpha=0.85, ncol=3)
plt.tight_layout(); plt.show()

## 7. 2D motion trajectories

Knee, wrist and COM paths overlaid on the **first sampled frame**, with fading trail (older = darker).

In [ ]:
def fading_line(ax, xs, ys, base_color, label=None):
    """Draw a line with alpha fading from old (low) → new (high)."""
    pts = np.array([xs, ys]).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    n = len(segs)
    if n == 0: return
    alphas = np.linspace(0.15, 0.95, n)
    for s, a in zip(segs, alphas):
        ax.plot(s[:,0], s[:,1], color=base_color, alpha=a, lw=1.8)
    if label:
        ax.plot(xs[-1], ys[-1], 'o', color=base_color, label=label, ms=8,
                markeredgecolor='white')

img = cv2.cvtColor(raw[0], cv2.COLOR_BGR2RGB)
h_, w_ = img.shape[:2]

fig, ax = plt.subplots(figsize=(9, 11))
ax.imshow(img, alpha=0.35)
for jname, color in [('left_knee','#4ECDC4'),('right_knee','#45B7D1'),
                     ('left_wrist','#FFA500'),('right_wrist','#FF6B6B')]:
    xs = [a[KEYPOINT[jname],0]*w_ for a in lm_arrs]
    ys = [a[KEYPOINT[jname],1]*h_ for a in lm_arrs]
    fading_line(ax, xs, ys, color, label=jname.replace('_',' '))

com_xs = [r['com'].com.x*w_ for r in results if r['com']]
com_ys = [r['com'].com.y*h_ for r in results if r['com']]
fading_line(ax, com_xs, com_ys, '#FFD700', label='COM')

ax.set_title(f'2D trajectory trails — {VIDEO_PATH.name}\n'
             f'(n={len(results)} sampled frames, brighter = newer)')
ax.legend(loc='upper right', framealpha=0.9)
ax.axis('off')
plt.tight_layout(); plt.show()

## 8. What exercise are you doing?

Friendly view of the heuristic detector. Left: total time spent in each
exercise (donut). Right: how confident the system was in each exercise
moment-by-moment (stacked area).

In [ ]:
from collections import Counter

EX_INFO = {
    'squat':    {'icon': '🏋️',  'label': 'Squat',    'color': '#4ECDC4'},
    'push_up':  {'icon': '💪',  'label': 'Push-up',  'color': '#FF6B6B'},
    'lunge':    {'icon': '🦵',  'label': 'Lunge',    'color': '#FFA500'},
    'deadlift': {'icon': '🔩',  'label': 'Deadlift', 'color': '#FFD700'},
    'unknown':  {'icon': '❓',  'label': 'Unknown',  'color': '#888888'},
}

# Headline summary
ex_counts = Counter(r['ex'].exercise for r in results)
top_ex, top_cnt = ex_counts.most_common(1)[0]
top_pct = 100.0 * top_cnt / len(results)
mean_conf = float(np.mean([r['ex'].exercise_confidence for r in results]))
info = EX_INFO.get(top_ex, EX_INFO['unknown'])

print(f"\n  {info['icon']}  Detected: {info['label']}")
print(f"      {top_pct:.0f}% of the clip   |   avg confidence {mean_conf*100:.0f}%\n")

# Two-panel figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5),
                               gridspec_kw={'width_ratios':[1, 1.6]})

# Donut: time per exercise
labels, sizes, colors = [], [], []
for ex_name, cnt in ex_counts.most_common():
    inf = EX_INFO.get(ex_name, EX_INFO['unknown'])
    pct = 100.0 * cnt / len(results)
    labels.append(f"{inf['icon']} {inf['label']}\n{pct:.0f}%")
    sizes.append(cnt)
    colors.append(inf['color'])

ax1.pie(sizes, labels=labels, colors=colors, startangle=90,
        wedgeprops=dict(width=0.45, edgecolor='#0f1724', linewidth=3),
        textprops=dict(color='#e0e0e0', fontsize=11, weight='bold'))
ax1.set_title('Time in each exercise', fontsize=13, pad=15)
ax1.text(0, 0, f"{info['icon']}\n{info['label']}",
         ha='center', va='center', fontsize=18, weight='bold',
         color=info['color'])

# Stacked area: confidence over time, normalised so it always fills 0-1
ex_order = ['squat','push_up','lunge','deadlift']
score_mat = np.array([[r['ex'].candidate_scores.get(e, 0.0) for e in ex_order]
                      for r in results])
row_sum = score_mat.sum(axis=1, keepdims=True); row_sum[row_sum < 1e-6] = 1.0
norm = score_mat / row_sum
x = np.arange(len(results))

ax2.stackplot(x, norm.T,
              labels=[f"{EX_INFO[e]['icon']} {EX_INFO[e]['label']}" for e in ex_order],
              colors=[EX_INFO[e]['color'] for e in ex_order],
              alpha=0.85)
ax2.set_xlim(0, len(results)-1); ax2.set_ylim(0, 1)
ax2.set_xlabel('Time (sampled frames)')
ax2.set_yticks([0, 0.5, 1.0]); ax2.set_yticklabels(['0%','50%','100%'])
ax2.set_title('Exercise mix at each moment', fontsize=13, pad=15)
ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.10),
           ncol=4, framealpha=0.0, fontsize=10)
ax2.grid(False)

plt.tight_layout()
plt.show()


## 9. How was your form?

Top: four KPI cards summarising the whole clip. Middle: traffic-light
timeline showing the worst-issue status per moment. Bottom: which body
parts triggered warnings most often (a quick "what to focus on next").

In [ ]:
from collections import Counter

# Severity per frame
sev_map = {STATUS_CRITICAL:3, STATUS_WARNING:2, STATUS_MONITOR:1, STATUS_OK:0}
sev_colors = {3:'#FF4444', 2:'#FFA500', 1:'#FFD700', 0:'#00D4AA'}
sev_emoji  = {3:'🛑',     2:'⚠️',     1:'👀',     0:'✅'}

max_sev = [(max((sev_map.get(f.status, 0) for f in r['qf']), default=0))
           for r in results]
total = max(1, len(max_sev))
counts = {s: sum(1 for v in max_sev if v == s) for s in (0,1,2,3)}
pcts   = {s: 100.0 * counts[s] / total for s in counts}

# Layout: 4 KPI cards on top row, timeline below, top issues at bottom
fig = plt.figure(figsize=(14, 8))
gs  = fig.add_gridspec(3, 4, height_ratios=[1.2, 0.7, 1.6],
                       hspace=0.55, wspace=0.25)

# KPI cards
order = [(0,'Clean','#00D4AA'), (1,'Watch','#FFD700'),
         (2,'Warning','#FFA500'), (3,'Correct now','#FF4444')]
for i, (sev, name, color) in enumerate(order):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor('#15202b')
    ax.add_patch(plt.Rectangle((0,0), 1, 1, transform=ax.transAxes,
                               facecolor='#15202b',
                               edgecolor=color, linewidth=2))
    ax.text(0.5, 0.78, sev_emoji[sev], ha='center', va='center',
            fontsize=30, transform=ax.transAxes)
    ax.text(0.5, 0.42, f"{pcts[sev]:.0f}%", ha='center', va='center',
            fontsize=26, weight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.15, f"{name} ({counts[sev]} frames)",
            ha='center', va='center',
            fontsize=10, color='#a0a0a0', transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.grid(False)

# Traffic-light timeline
ax_bar = fig.add_subplot(gs[1, :])
ax_bar.bar(range(len(max_sev)), [1]*len(max_sev),
           color=[sev_colors[s] for s in max_sev], width=1.0)
ax_bar.set_xlim(-0.5, len(max_sev)-0.5); ax_bar.set_ylim(0, 1)
ax_bar.set_yticks([]); ax_bar.set_xlabel('Time (sampled frames)')
ax_bar.set_title('Form quality timeline  ✅ Clean → 👀 Watch → ⚠️ Warning → 🛑 Correct now',
                 fontsize=12, pad=10)
ax_bar.grid(False)
for spine in ['top','right','left']: ax_bar.spines[spine].set_visible(False)

# Top issues — count flagged joints across all frames
issue_counter = Counter()
for r in results:
    for f in r['qf']:
        if f.status in (STATUS_WARNING, STATUS_CRITICAL):
            label = f.joint or f.id.replace('rule_','rule ').replace('_',' ')
            issue_counter[label] += 1

ax_top = fig.add_subplot(gs[2, :])
if issue_counter:
    items = issue_counter.most_common(6)
    names = [k.replace('_',' ').title() for k, _ in items]
    vals  = [v for _, v in items]
    bars = ax_top.barh(names[::-1], vals[::-1],
                       color='#FF6B6B', edgecolor='#FF4444')
    for bar, v in zip(bars, vals[::-1]):
        ax_top.text(bar.get_width() + max(vals)*0.02,
                    bar.get_y() + bar.get_height()/2,
                    f' {v}', va='center', color='#e0e0e0', fontsize=10)
    ax_top.set_xlabel('Frames flagged')
    ax_top.set_title('What to focus on — most-flagged body parts',
                     fontsize=12, pad=10)
    ax_top.grid(False, axis='y')
    for spine in ['top','right']: ax_top.spines[spine].set_visible(False)
else:
    ax_top.text(0.5, 0.5, '🎉  No warnings detected — great form!',
                ha='center', va='center', fontsize=18, color='#00D4AA',
                transform=ax_top.transAxes)
    ax_top.axis('off')

plt.suptitle(f'Form quality summary — {VIDEO_PATH.name}',
             fontsize=14, color='#00D4AA', y=0.995)
plt.show()

# Plain-language verdict
if pcts[3] > 5:
    verdict = f"🛑  {pcts[3]:.0f}% of frames need correction. Slow down and review the flagged joints."
elif pcts[2] > 15:
    verdict = f"⚠️  {pcts[2]:.0f}% of frames had warnings. Focus on the top issue listed above."
elif pcts[1] > 30:
    verdict = f"👀  Mostly clean, but {pcts[1]:.0f}% of frames were borderline. Stay focused on form."
else:
    verdict = f"✅  Clean clip — {pcts[0]:.0f}% of frames were OK. Keep it up."
print('\n' + verdict)


## 10. Single-frame deep dive

In [ ]:
PICK_FRAME = len(results) // 2          # ← change to inspect any frame
r = results[PICK_FRAME]

print(f"=== Frame {PICK_FRAME} ===")
print(f"Exercise:    {r['ex'].exercise}  ({r['ex'].exercise_confidence:.2f})")
print(f"Phase:       {r['phase']}")
print(f"Rep count:   {r['rep']}")
print()
print('Template metrics:')
for k, v in r['metrics'].items(): print(f'  {k:30s} {v}')
print()
print('Quality flags:')
if r['qf']:
    for f in r['qf']:
        print(f"  [{f.status:9s}] rule {f.rule}: {f.id}  joint={f.joint}  value={f.value}")
else:
    print('  (none)')
print()
print('Not-applicable rules:')
if r['na']:
    for n in r['na']:
        print(f"  rule {n.rule}: {n.id}  → {n.reason}")
else:
    print('  (none)')
print()
print('Mock Gemma message:')
print(f"  [{r['gemma']['level'].upper()}] {r['gemma']['message']}")
print(f"  source: {r['gemma']['source']}")
print(f"  safety: {r['gemma']['safety_note']}")

## 11. (Optional) Real Gemma 4 inference for the picked frame

Requires `llama-cpp-python` and a local GGUF in `models/`. Skip if running on minimal env.

In [ ]:
USE_GEMMA = False    # ← set True to call local Gemma 4
MODEL = 'gemma4-e2b-Q4_K_M.gguf'   # E2B is faster (~5s); E4B is higher quality (~15s)

if USE_GEMMA:
    from gemma_local import gemma_feedback, HAS_LLAMA
    if not HAS_LLAMA:
        print('llama-cpp-python not installed; skipping Gemma call.')
    else:
        path = ROOT_DIR / 'models' / MODEL
        out = gemma_feedback(r['report'], str(path), max_tokens=60)
        print(f"[{out['level'].upper()}] {out['message']}")
        print(f"source: {out['source']}")
else:
    print('Gemma disabled. Set USE_GEMMA=True to enable.')

## 12. Export structured report JSON

In [ ]:
OUT_DIR = PROTO_DIR / 'data' / 'reports'
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / f'{VIDEO_PATH.stem}_report.json'

payload = {
    'video': VIDEO_PATH.name,
    'fps': fps, 'sample_every': SAMPLE_EVERY,
    'n_frames': len(results),
    'dominant_exercise': ex_dist.most_common(1)[0][0],
    'frames': [r['report'].to_dict() for r in results],
}
with open(out_path, 'w', encoding='utf-8') as fh:
    json.dump(payload, fh, ensure_ascii=False, indent=2)
print(f'Wrote {out_path}  ({out_path.stat().st_size/1024:.1f} KB)')